In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymer4.models import Lmer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_recall_curve,
    roc_curve,
    fbeta_score,
    auc
)
from sklearn.feature_selection import (
    SelectFromModel,
    VarianceThreshold
)
from sklearn.model_selection import (
    ParameterGrid,
    GridSearchCV,
    RandomizedSearchCV,
    GroupKFold,
    GroupShuffleSplit
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.utils.class_weight import compute_class_weight
from IOHMM import UnSupervisedIOHMM
from IOHMM.linear_models import GLM, DiscreteMNL, CrossEntropyMNL
import optuna

from ingestion.load_openesm import load_curated_dataset

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
sns.set_style("whitegrid")

In [3]:
df = load_curated_dataset("0006")

In [4]:
df.head()

,id,day,beep,responded,day_index,beep_index,prompt_index,day_squared,beep_squared,day_beep_interaction,...,prior_consecutive_active_days,prior_consecutive_inactive_days,prior_response_rate_change,prior_3_day_response_rate,prior_3_day_participation_rate,prior_7_day_response_rate,prior_7_day_participation_rate,prior_ewm_3_day_response_rate,prior_ewm_6_day_response_rate,missed_day
0,1,1,1,1,1,1,1,1,1,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,1,2,0,1,2,2,1,4,2,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1,1,3,1,1,3,3,1,9,3,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,1,1,4,1,1,4,4,1,16,4,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1,1,5,1,1,5,5,1,25,5,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [5]:
# Dataset Dimensions
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Participants: {df['id'].nunique():,}")
print(f"Study Days: {df['day'].nunique():,}")

# Data Types and Non-Null Counts
display(
    df.dtypes.rename("dtype")
    .to_frame()
    .join(df.count().rename("non_null"))
    .assign(
        missing=lambda x: len(df) - x["non_null"],
        missing_pct=lambda x: (
            (len(df) - x["non_null"]) / len(df) * 100
        ).round(2),
        unique=lambda x: df.nunique()
    )
    .sort_values("missing_pct", ascending=False)
)

# Target Distribution
if "responded" in df.columns:
    print("\nTarget Distribution:")
    display(
        df["responded"]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
        .assign(
            percentage=lambda x: (
                x["count"] / len(df) * 100
            ).round(2)
        )
    )

# Participant-Level Distribution
print("\nRows per Participant:")
display(
    df.groupby("id", dropna=False)
    .size()
    .describe()
    .to_frame("rows_per_participant")
)

# Sample Records
display(df.head())

Rows: 30,000
Columns: 92
Participants: 125
Study Days: 40


,dtype,non_null,missing,missing_pct,unique
beep_gap_from_previous_same_day,Int64,25000,5000,16.67,1
prior_same_day_3_prompt_response_rate,float64,25000,5000,16.67,5
prior_same_day_ewm_3_response_rate,float64,25000,5000,16.67,17
prev_response_same_day,float64,25000,5000,16.67,2
prior_same_day_response_rate,float64,25000,5000,16.67,11
...,...,...,...,...,...
prior_days_with_response,float64,30000,0,0.00,40
prior_consecutive_inactive_days,int64,30000,0,0.00,9
prior_consecutive_active_days,int64,30000,0,0.00,40
prior_days_observed,int64,30000,0,0.00,40



Target Distribution:


,count,percentage
responded,,
1,21570,71.9
0,8430,28.1



Rows per Participant:


,rows_per_participant
count,125.0
mean,240.0
std,0.0
min,240.0
25%,240.0
50%,240.0
75%,240.0
max,240.0


,id,day,beep,responded,day_index,beep_index,prompt_index,day_squared,beep_squared,day_beep_interaction,...,prior_consecutive_active_days,prior_consecutive_inactive_days,prior_response_rate_change,prior_3_day_response_rate,prior_3_day_participation_rate,prior_7_day_response_rate,prior_7_day_participation_rate,prior_ewm_3_day_response_rate,prior_ewm_6_day_response_rate,missed_day
0,1,1,1,1,1,1,1,1,1,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,1,2,0,1,2,2,1,4,2,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1,1,3,1,1,3,3,1,9,3,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,1,1,4,1,1,4,4,1,16,4,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1,1,5,1,1,5,5,1,25,5,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [8]:
# Feature Audit
feature_audit = pd.DataFrame({
    "dtype": df.dtypes,
    "unique": df.nunique(dropna=True),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
})

feature_audit["unique_pct"] = (
    feature_audit["unique"] / len(df) * 100
)

feature_audit["constant"] = feature_audit["unique"] <= 1

feature_audit = feature_audit.sort_values(
    ["constant", "missing_pct", "unique"],
    ascending=[False, False, True]
)

display(feature_audit)

,dtype,unique,missing,missing_pct,unique_pct,constant
beep_gap_from_previous_same_day,Int64,1,5000,16.67,0.003333,True
days_since_previous_same_beep,Int64,1,750,2.50,0.003333,True
days_since_previous_observed_day,Int64,1,750,2.50,0.003333,True
prompts_in_day,Int64,1,0,0.00,0.003333,True
scheduled_beeps_in_day,Int64,1,0,0.00,0.003333,True
prev_response_same_day,float64,2,5000,16.67,0.006667,False
prior_same_day_2_prompt_response_rate,float64,3,5000,16.67,0.010000,False
prior_same_day_3_prompt_response_rate,float64,5,5000,16.67,0.016667,False
prior_same_day_response_rate,float64,11,5000,16.67,0.036667,False
prior_same_day_ewm_3_response_rate,float64,17,5000,16.67,0.056667,False


In [9]:
TARGET = "nonresponse"

df["nonresponse"] = (1 - df["responded"]).astype("int8")


# Structural Columns
STRUCTURAL_COLUMNS = [
    "id",
    "day",
    "beep",
]


# Target and Retrospective Columns
EXCLUDED_COLUMNS = [
    "responded",
    "responded_any_today",
    "daily_response_rate",
    "missed_day",
    "cumulative_responses",
    "cumulative_prompts",
    "cumulative_response_rate",
    "last_observed_day",
]


# Constant Columns
CONSTANT_COLUMNS = [
    "beep_gap_from_previous_same_day",
    "days_since_previous_same_beep",
    "days_since_previous_observed_day",
    "prompts_in_day",
    "scheduled_beeps_in_day",
]


# Remove Constant and Excluded Columns
columns_to_remove = CONSTANT_COLUMNS + EXCLUDED_COLUMNS

model_df = df.drop(
    columns=columns_to_remove,
    errors="ignore"
).copy()


print(f"Original columns: {len(df.columns)}")
print(f"Remaining columns: {len(model_df.columns)}")
print(f"Predictor candidates: {len(model_df.columns) - 1 - len(STRUCTURAL_COLUMNS)}")

display(model_df.head())

Original columns: 93
Remaining columns: 80
Predictor candidates: 76


,id,day,beep,day_index,beep_index,prompt_index,day_squared,beep_squared,day_beep_interaction,days_since_first_observed,first_observed_day,first_beep_of_day,last_beep_of_day,beep_position,days_since_previous_prompt,beep_difference_from_previous,scheduled_beep_position,scheduled_beep_position_rate,prior_cumulative_responses,prior_cumulative_prompts,prior_cumulative_response_rate,prior_cumulative_nonresponses,prior_missed_beep_count,prior_smoothed_response_rate,prior_response_history_count,prev_1_response,prev_2_response,prev_3_response,prev_4_response,prev_5_response,prev_6_response,prev_response,prev_response_same_day,prior_3_prompt_response_rate,prior_6_prompt_response_rate,prior_12_prompt_response_rate,prior_ewm_3_prompt_response_rate,prior_ewm_6_prompt_response_rate,prior_response_streak,prior_nonresponse_streak,days_since_previous_response,prior_same_day_responses,prior_same_day_prompts,prior_same_day_nonresponses,prior_same_day_response_rate,prior_same_day_smoothed_response_rate,prior_same_day_response_streak,prior_same_day_nonresponse_streak,prior_same_day_2_prompt_response_rate,prior_same_day_3_prompt_response_rate,prior_same_day_ewm_3_response_rate,prior_same_day_ewm_6_response_rate,prior_same_beep_responses,prior_same_beep_observations,prior_same_beep_nonresponses,prior_same_beep_response_rate,prior_same_beep_smoothed_rate,prior_same_beep_response_streak,prior_same_beep_nonresponse_streak,prev_same_beep_response,prior_same_beep_ewm_3_response_rate,prior_same_beep_ewm_6_response_rate,prev_day_same_beep_response,prev_day_same_beep_available,prior_day_response_rate,prior_days_with_response,prior_days_observed,prior_day_participation_rate,prev_day_responded,prev_day_response_rate,prior_consecutive_active_days,prior_consecutive_inactive_days,prior_response_rate_change,prior_3_day_response_rate,prior_3_day_participation_rate,prior_7_day_response_rate,prior_7_day_participation_rate,prior_ewm_3_day_response_rate,prior_ewm_6_day_response_rate,nonresponse
0,1,1,1,1,1,1,1,1,1,0,1,1,0,0.166667,<NA>,<NA>,1,0.166667,0,0,NaN,0,0,50.000000,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,<NA>,0,0,0,NaN,50.000000,0,0,NaN,NaN,NaN,NaN,0,0,0,NaN,50.0,0,0,NaN,NaN,NaN,NaN,0,NaN,0.0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,1,2,1,2,2,1,4,2,0,1,0,0,0.333333,0,1,2,0.333333,1,1,100.000000,0,0,66.666667,1,1.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,100.000000,100.000000,100.000000,100.0,100.000000,1,0,0,1,1,0,100.000000,66.666667,1,0,100.0,100.000000,100.0,100.000000,0,0,0,NaN,50.0,0,0,NaN,NaN,NaN,NaN,0,NaN,0.0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,1,1,3,1,3,3,1,9,3,0,1,0,0,0.5,0,1,3,0.5,1,2,50.000000,1,1,50.000000,2,0.0,1.0,NaN,NaN,NaN,NaN,0.0,0.0,50.000000,50.000000,50.000000,50.0,71.428571,0,1,0,1,2,1,50.000000,50.000000,0,1,50.0,50.000000,50.0,71.428571,0,0,0,NaN,50.0,0,0,NaN,NaN,NaN,NaN,0,NaN,0.0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,1,1,4,1,4,4,1,16,4,0,1,0,0,0.666667,0,1,4,0.666667,2,3,66.666667,1,1,60.000000,3,1.0,0.0,1.0,NaN,NaN,NaN,1.0,1.0,66.666667,66.666667,66.666667,75.0,79.591837,1,0,0,2,3,1,66.666667,60.000000,1,0,50.0,66.666667,75.0,79.591837,0,0,0,NaN,50.0,0,0,NaN,NaN,NaN,NaN,0,NaN,0.0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1,1,5,1,5,5,1,25,5,0,1,0,0,0.833333,0,1,5,0.833333,3,4,75.000000,1,1,66.666667,4,1.0,1.0,0.0,1.0,NaN,NaN,1.0,1.0,66.666667,75.000000,75.000000,87.5,85.422741,2,0,0,3,4,1,75.000000,66.666667,2,0,100.0,66.666667,87.5,85.422741,0,0,0,NaN,50.0,0,0,NaN,NaN,NaN,NaN,0,NaN,0.0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [10]:
# Find Exact Duplicate Columns
duplicate_pairs = []

columns = list(model_df.columns)

for i, column in enumerate(columns):
    for other_column in columns[i + 1:]:
        if model_df[column].equals(model_df[other_column]):
            duplicate_pairs.append((column, other_column))

print("Exact duplicate feature pairs:")

for column, other_column in duplicate_pairs:
    print(f"{column} == {other_column}")

Exact duplicate feature pairs:
day == day_index
beep == beep_index
beep_position == scheduled_beep_position_rate
prior_cumulative_prompts == prior_response_history_count
prior_cumulative_nonresponses == prior_missed_beep_count
prev_1_response == prev_response
prev_6_response == prev_same_beep_response
prev_6_response == prev_day_same_beep_response
prior_same_beep_observations == prior_days_observed
prev_same_beep_response == prev_day_same_beep_response


In [11]:
# ============================================================
# 1. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

TARGET = "nonresponse"
PARTICIPANT_COL = "id"

# Remove exact duplicate features identified in the audit
duplicate_columns_to_drop = sorted({
    duplicate[1]
    for duplicate in duplicate_pairs
})

model_df = model_df.drop(
    columns=duplicate_columns_to_drop,
    errors="ignore"
).copy()

# Chronological partitions
train_df = model_df[
    model_df["day"] <= 28
].copy()

validation_df = model_df[
    (model_df["day"] >= 29) &
    (model_df["day"] <= 34)
].copy()

test_df = model_df[
    model_df["day"] >= 35
].copy()

# Check partition sizes
print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nTarget distribution:")

for name, partition in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print(partition[TARGET].value_counts(normalize=True).round(4))

Training: (21000, 71)
Validation: (4500, 71)
Test: (4500, 71)

Target distribution:

Training
nonresponse
0    0.7247
1    0.2753
Name: proportion, dtype: float64

Validation
nonresponse
0    0.7007
1    0.2993
Name: proportion, dtype: float64

Test
nonresponse
0    0.7109
1    0.2891
Name: proportion, dtype: float64


In [12]:
# ============================================================
# 2. DEFINE X AND Y
# ============================================================

NON_PREDICTOR_COLUMNS = [
    TARGET,
    "id",
    "responded",
]

X_train = train_df.drop(
    columns=NON_PREDICTOR_COLUMNS,
    errors="ignore"
)

X_validation = validation_df.drop(
    columns=NON_PREDICTOR_COLUMNS,
    errors="ignore"
)

X_test = test_df.drop(
    columns=NON_PREDICTOR_COLUMNS,
    errors="ignore"
)

y_train = train_df[TARGET].astype(int)
y_validation = validation_df[TARGET].astype(int)
y_test = test_df[TARGET].astype(int)

# Ensure consistent predictor ordering
X_validation = X_validation[X_train.columns]
X_test = X_test[X_train.columns]

# Convert nullable pandas numeric columns to standard floats.
# This also ensures missing values are represented as np.nan.
X_train = X_train.astype("float64")
X_validation = X_validation.astype("float64")
X_test = X_test.astype("float64")

print("Predictor count:", X_train.shape[1])
print("Training observations:", len(X_train))
print("Validation observations:", len(X_validation))
print("Test observations:", len(X_test))

Predictor count: 69
Training observations: 21000
Validation observations: 4500
Test observations: 4500


# Regularized Logistic

In [15]:
# ============================================================
# 4. MODEL PIPELINE
# ============================================================

def create_logistic_pipeline(
    penalty="l2",
    C=1.0,
    l1_ratio=None
):
    """
    Create a preprocessing and logistic regression pipeline.
    """

    model_parameters = {
        "penalty": penalty,
        "C": C,
        "max_iter": 10000,
        "random_state": 42,
        "class_weight": None,
    }

    if penalty == "elasticnet":
        model_parameters["solver"] = "saga"
        model_parameters["l1_ratio"] = l1_ratio

    elif penalty == "l1":
        model_parameters["solver"] = "saga"

    else:
        model_parameters["solver"] = "lbfgs"

    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(**model_parameters)
        )
    ])

    return pipeline

In [16]:
# ============================================================
# 5. HYPERPARAMETER TUNING
# ============================================================

C_VALUES = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
]

PARAMETER_CONFIGURATIONS = []

# L2
for C in C_VALUES:
    PARAMETER_CONFIGURATIONS.append({
        "penalty": "l2",
        "C": C,
        "l1_ratio": None,
    })

# L1
for C in C_VALUES:
    PARAMETER_CONFIGURATIONS.append({
        "penalty": "l1",
        "C": C,
        "l1_ratio": None,
    })

# Elastic Net
for C in C_VALUES:
    for l1_ratio in [0.25, 0.5, 0.75]:
        PARAMETER_CONFIGURATIONS.append({
            "penalty": "elasticnet",
            "C": C,
            "l1_ratio": l1_ratio,
        })


tuning_results = []
fitted_models = {}

for i, parameters in enumerate(
    PARAMETER_CONFIGURATIONS,
    start=1
):

    print(
        f"Fitting model {i}/{len(PARAMETER_CONFIGURATIONS)}: "
        f"{parameters}"
    )

    pipeline = create_logistic_pipeline(
        penalty=parameters["penalty"],
        C=parameters["C"],
        l1_ratio=parameters["l1_ratio"],
    )

    pipeline.fit(
        X_train,
        y_train
    )

    validation_probabilities = pipeline.predict_proba(
        X_validation
    )[:, 1]

    ap = average_precision_score(
        y_validation,
        validation_probabilities
    )

    validation_logloss = log_loss(
        y_validation,
        validation_probabilities,
        labels=[0, 1]
    )

    brier = brier_score_loss(
        y_validation,
        validation_probabilities
    )

    result = {
        **parameters,
        "average_precision": ap,
        "log_loss": validation_logloss,
        "brier_score": brier,
    }

    tuning_results.append(result)

    fitted_models[i] = pipeline


# Results table
tuning_results_df = pd.DataFrame(
    tuning_results
).sort_values(
    ["average_precision", "log_loss"],
    ascending=[False, True]
).reset_index(drop=True)

display(tuning_results_df)

Fitting model 1/30: {'penalty': 'l2', 'C': 0.001, 'l1_ratio': None}
Fitting model 2/30: {'penalty': 'l2', 'C': 0.01, 'l1_ratio': None}
Fitting model 3/30: {'penalty': 'l2', 'C': 0.1, 'l1_ratio': None}
Fitting model 4/30: {'penalty': 'l2', 'C': 1.0, 'l1_ratio': None}
Fitting model 5/30: {'penalty': 'l2', 'C': 10.0, 'l1_ratio': None}
Fitting model 6/30: {'penalty': 'l2', 'C': 100.0, 'l1_ratio': None}
Fitting model 7/30: {'penalty': 'l1', 'C': 0.001, 'l1_ratio': None}
Fitting model 8/30: {'penalty': 'l1', 'C': 0.01, 'l1_ratio': None}
Fitting model 9/30: {'penalty': 'l1', 'C': 0.1, 'l1_ratio': None}
Fitting model 10/30: {'penalty': 'l1', 'C': 1.0, 'l1_ratio': None}
Fitting model 11/30: {'penalty': 'l1', 'C': 10.0, 'l1_ratio': None}
Fitting model 12/30: {'penalty': 'l1', 'C': 100.0, 'l1_ratio': None}
Fitting model 13/30: {'penalty': 'elasticnet', 'C': 0.001, 'l1_ratio': 0.25}
Fitting model 14/30: {'penalty': 'elasticnet', 'C': 0.001, 'l1_ratio': 0.5}
Fitting model 15/30: {'penalty': 'elasti

,penalty,C,l1_ratio,average_precision,log_loss,brier_score
0,elasticnet,0.100,0.75,0.568351,0.530034,0.175793
1,elasticnet,0.100,0.50,0.568228,0.530312,0.175893
2,l1,0.100,NaN,0.568160,0.529862,0.175728
3,l2,0.010,NaN,0.568157,0.529827,0.175727
4,elasticnet,0.100,0.25,0.567985,0.530776,0.176062
5,l2,0.100,NaN,0.567955,0.531660,0.176411
6,l2,1.000,NaN,0.567624,0.532564,0.176711
7,l2,10.000,NaN,0.567623,0.532616,0.176714
8,elasticnet,1.000,0.75,0.567530,0.532231,0.176584
9,elasticnet,1.000,0.25,0.567500,0.532489,0.176682


In [17]:
# ============================================================
# 6. SELECT BEST MODEL
# ============================================================
best_parameters = tuning_results_df.iloc[0].to_dict()

best_model_index = next(
    i
    for i, parameters in enumerate(
        PARAMETER_CONFIGURATIONS,
        start=1
    )
    if (
        parameters["penalty"] == best_parameters["penalty"]
        and parameters["C"] == best_parameters["C"]
        and parameters["l1_ratio"] == best_parameters["l1_ratio"]
    )
)

best_model = fitted_models[best_model_index]

print("Best parameters:")
print({
    "penalty": best_parameters["penalty"],
    "C": best_parameters["C"],
    "l1_ratio": best_parameters["l1_ratio"],
})


# Validation probabilities
validation_probabilities = best_model.predict_proba(
    X_validation
)[:, 1]

Best parameters:
{'penalty': 'elasticnet', 'C': 0.1, 'l1_ratio': 0.75}


In [18]:
# ============================================================
# 7. OPTIMAL CLASSIFICATION THRESHOLD
# ============================================================

precision, recall, thresholds = precision_recall_curve(
    y_validation,
    validation_probabilities
)

# F2 calculation for each available threshold
f2_scores = (
    5 * precision[:-1] * recall[:-1]
    /
    (
        4 * precision[:-1] + recall[:-1] + 1e-10
    )
)

best_threshold_index = np.argmax(f2_scores)

best_threshold = thresholds[best_threshold_index]

print(f"Optimal validation threshold: {best_threshold:.4f}")
print(f"Validation F2: {f2_scores[best_threshold_index]:.4f}")
print(f"Validation precision: {precision[best_threshold_index]:.4f}")
print(f"Validation recall: {recall[best_threshold_index]:.4f}")

Optimal validation threshold: 0.1305
Validation F2: 0.7004
Validation precision: 0.3497
Validation recall: 0.9347


In [22]:
# ============================================================
# 8. VALIDATION EVALUATION
# ============================================================
def evaluate_model(
    y_true,
    probabilities,
    threshold=0.5,
    dataset_name="Model",
):
    """
    Evaluate binary nonresponse predictions.

    Positive class:
        1 = Nonresponse

    Parameters
    ----------
    y_true : array-like
        Actual binary outcomes.

    probabilities : array-like
        Predicted probability of nonresponse.

    threshold : float
        Probability threshold for classifying nonresponse.

    dataset_name : str
        Label for the evaluation output.
    """

    y_pred = (probabilities >= threshold).astype(int)

    metrics = {
        "dataset": dataset_name,
        "threshold": threshold,
        "average_precision": average_precision_score(
            y_true, probabilities
        ),
        "roc_auc": roc_auc_score(
            y_true, probabilities
        ),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f2": fbeta_score(
            y_true, y_pred, beta=2, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "mcc": matthews_corrcoef(
            y_true, y_pred
        ),
        "brier_score": brier_score_loss(
            y_true, probabilities
        ),
        "log_loss": log_loss(
            y_true, probabilities
        ),
    }

    print(f"\n{'=' * 55}")
    print(f"{dataset_name.upper()} EVALUATION")
    print(f"{'=' * 55}")

    print(f"Threshold: {threshold:.4f}")

    print("\nMetrics:")
    display(
        pd.DataFrame([metrics])
        .set_index("dataset")
        .round(4)
    )

    print("\nConfusion Matrix:")
    print("Rows = Actual | Columns = Predicted")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["Responded", "Nonresponse"],
            zero_division=0,
        )
    )

    return metrics


validation_metrics = evaluate_model(
    y_true=y_validation,
    probabilities=validation_probabilities,
    threshold=best_threshold,
    dataset_name="Validation"
)


VALIDATION EVALUATION
Threshold: 0.1305

Metrics:


,threshold,average_precision,roc_auc,precision,recall,f2,balanced_accuracy,mcc,brier_score,log_loss
dataset,,,,,,,,,,
Validation,0.1305,0.5684,0.7393,0.3497,0.9347,0.7004,0.5961,0.2201,0.1758,0.53



Confusion Matrix:
Rows = Actual | Columns = Predicted
[[ 812 2341]
 [  88 1259]]

Classification Report:
              precision    recall  f1-score   support

   Responded       0.90      0.26      0.40      3153
 Nonresponse       0.35      0.93      0.51      1347

    accuracy                           0.46      4500
   macro avg       0.63      0.60      0.45      4500
weighted avg       0.74      0.46      0.43      4500



In [24]:
# ============================================================
# 9. FINAL MODEL FIT
# ============================================================

X_train_validation = pd.concat(
    [X_train, X_validation],
    axis=0
)

y_train_validation = pd.concat(
    [y_train, y_validation],
    axis=0
)

final_model = create_logistic_pipeline(
    penalty=best_parameters["penalty"],
    C=best_parameters["C"],
    l1_ratio=best_parameters["l1_ratio"],
)

final_model.fit(
    X_train_validation,
    y_train_validation
)


# Test predictions
test_probabilities = final_model.predict_proba(
    X_test
)[:, 1]


# Final test evaluation
test_metrics = evaluate_model(
    y_true=y_test,
    probabilities=test_probabilities,
    threshold=best_threshold,
    dataset_name="Test"
)


TEST EVALUATION
Threshold: 0.1305

Metrics:


,threshold,average_precision,roc_auc,precision,recall,f2,balanced_accuracy,mcc,brier_score,log_loss
dataset,,,,,,,,,,
Test,0.1305,0.5439,0.7252,0.341,0.9154,0.6847,0.5979,0.213,0.1765,0.5309



Confusion Matrix:
Rows = Actual | Columns = Predicted
[[ 897 2302]
 [ 110 1191]]

Classification Report:
              precision    recall  f1-score   support

   Responded       0.89      0.28      0.43      3199
 Nonresponse       0.34      0.92      0.50      1301

    accuracy                           0.46      4500
   macro avg       0.62      0.60      0.46      4500
weighted avg       0.73      0.46      0.45      4500



In [25]:
# ============================================================
# 10. FEATURE COEFFICIENTS
# ============================================================

fitted_logistic = final_model.named_steps["model"]

imputer = final_model.named_steps["imputer"]

# Original feature names plus generated missingness indicators
feature_names = imputer.get_feature_names_out(
    X_train.columns
)

coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": fitted_logistic.coef_[0],
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

display(coefficients)

print(
    "Nonzero coefficients:",
    (coefficients["coefficient"] != 0).sum()
)

print(
    "Zero coefficients:",
    (coefficients["coefficient"] == 0).sum()
)

,feature,coefficient,absolute_coefficient
65,prior_7_day_response_rate,-0.314029,0.314029
47,prior_same_beep_nonresponses,0.235773,0.235773
33,days_since_previous_response,0.228712,0.228712
3,day_squared,-0.187964,0.187964
18,prior_smoothed_response_rate,-0.171052,0.171052
61,prior_consecutive_inactive_days,-0.158583,0.158583
4,beep_squared,0.143668,0.143668
58,prev_day_responded,-0.137996,0.137996
25,prev_response_same_day,-0.126967,0.126967
39,prior_same_day_response_streak,-0.101513,0.101513


Nonzero coefficients: 69
Zero coefficients: 35


In [26]:
# Baseline: Constant Training Nonresponse Probability

baseline_probability = y_train.mean()

baseline_validation_probabilities = np.full(
    len(y_validation),
    baseline_probability
)

baseline_test_probabilities = np.full(
    len(y_test),
    baseline_probability
)

baseline_validation_metrics = evaluate_model(
    y_validation,
    baseline_validation_probabilities,
    threshold=0.5,
    dataset_name="Baseline Validation"
)

baseline_test_metrics = evaluate_model(
    y_test,
    baseline_test_probabilities,
    threshold=0.5,
    dataset_name="Baseline Test"
)


BASELINE VALIDATION EVALUATION
Threshold: 0.5000

Metrics:


,threshold,average_precision,roc_auc,precision,recall,f2,balanced_accuracy,mcc,brier_score,log_loss
dataset,,,,,,,,,,
Baseline Validation,0.5,0.2993,0.5,0.0,0.0,0.0,0.5,0.0,0.2103,0.6117



Confusion Matrix:
Rows = Actual | Columns = Predicted
[[3153    0]
 [1347    0]]

Classification Report:
              precision    recall  f1-score   support

   Responded       0.70      1.00      0.82      3153
 Nonresponse       0.00      0.00      0.00      1347

    accuracy                           0.70      4500
   macro avg       0.35      0.50      0.41      4500
weighted avg       0.49      0.70      0.58      4500


BASELINE TEST EVALUATION
Threshold: 0.5000

Metrics:


,threshold,average_precision,roc_auc,precision,recall,f2,balanced_accuracy,mcc,brier_score,log_loss
dataset,,,,,,,,,,
Baseline Test,0.5,0.2891,0.5,0.0,0.0,0.0,0.5,0.0,0.2057,0.6018



Confusion Matrix:
Rows = Actual | Columns = Predicted
[[3199    0]
 [1301    0]]

Classification Report:
              precision    recall  f1-score   support

   Responded       0.71      1.00      0.83      3199
 Nonresponse       0.00      0.00      0.00      1301

    accuracy                           0.71      4500
   macro avg       0.36      0.50      0.42      4500
weighted avg       0.51      0.71      0.59      4500



In [27]:
participant_test_results = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "actual": y_test.to_numpy(),
    "probability": test_probabilities,
})

participant_test_results["prediction"] = (
    participant_test_results["probability"] >= best_threshold
).astype(int)


participant_metrics = []

for participant_id, group in participant_test_results.groupby("id"):

    actual = group["actual"]
    probabilities = group["probability"]
    predictions = group["prediction"]

    participant_metrics.append({
        "id": participant_id,
        "prompts": len(group),
        "nonresponse_rate": actual.mean(),

        "average_precision": (
            average_precision_score(actual, probabilities)
            if actual.sum() > 0 else np.nan
        ),

        "roc_auc": (
            roc_auc_score(actual, probabilities)
            if actual.nunique() == 2 else np.nan
        ),

        "brier_score": brier_score_loss(
            actual, probabilities
        ),

        "precision": precision_score(
            actual, predictions, zero_division=0
        ),

        "recall": recall_score(
            actual, predictions, zero_division=0
        ),

        "f2": fbeta_score(
            actual, predictions, beta=2, zero_division=0
        ),
    })


participant_metrics_df = pd.DataFrame(participant_metrics)

display(
    participant_metrics_df.describe().round(4)
)

,id,prompts,nonresponse_rate,average_precision,roc_auc,brier_score,precision,recall,f2
count,125.000,125.0,125.0000,123.0000,123.0000,125.0000,125.0000,125.0000,125.0000
mean,80.736,36.0,0.2891,0.4018,0.5523,0.1765,0.2971,0.7935,0.5601
std,47.115,0.0,0.1832,0.2019,0.1378,0.0659,0.1930,0.3398,0.2666
min,1.000,36.0,0.0000,0.0278,0.0000,0.0054,0.0000,0.0000,0.0000
25%,37.000,36.0,0.1667,0.2459,0.4733,0.1265,0.1714,0.6667,0.4082
50%,84.000,36.0,0.2500,0.3842,0.5613,0.1835,0.2581,1.0000,0.6250
75%,121.000,36.0,0.3889,0.5260,0.6423,0.2332,0.3889,1.0000,0.7609
max,164.000,36.0,0.9167,0.9826,0.8500,0.2795,0.9167,1.0000,0.9821


In [28]:
display(
    participant_metrics_df.sort_values(
        "average_precision",
        ascending=True
    ).round(4)
)

,id,prompts,nonresponse_rate,average_precision,roc_auc,brier_score,precision,recall,f2
26,32,36,0.0278,0.0278,0.0000,0.0322,0.0000,0.0000,0.0000
11,14,36,0.0833,0.0660,0.1717,0.0815,0.0000,0.0000,0.0000
36,46,36,0.0556,0.0841,0.3971,0.0651,0.0417,0.5000,0.1562
31,37,36,0.1111,0.0851,0.2031,0.1049,0.0000,0.0000,0.0000
120,156,36,0.1111,0.0880,0.2109,0.1059,0.0526,0.2500,0.1429
21,27,36,0.0833,0.0996,0.4747,0.0772,0.0000,0.0000,0.0000
23,29,36,0.0278,0.1000,0.7429,0.0471,0.0667,1.0000,0.2632
4,6,36,0.1111,0.1164,0.4219,0.1118,0.1176,1.0000,0.4000
43,55,36,0.1389,0.1359,0.3935,0.1301,0.1212,0.8000,0.3774
66,93,36,0.0833,0.1369,0.4949,0.0781,0.0000,0.0000,0.0000


In [29]:
participant_baseline = (
    train_df.groupby("id")["nonresponse"]
    .mean()
    .rename("baseline_probability")
)

participant_test_results["baseline_probability"] = (
    participant_test_results["id"]
    .map(participant_baseline)
)

baseline_brier = brier_score_loss(
    participant_test_results["actual"],
    participant_test_results["baseline_probability"],
)

model_brier = brier_score_loss(
    participant_test_results["actual"],
    participant_test_results["probability"],
)

print(f"Participant baseline Brier: {baseline_brier:.4f}")
print(f"Logistic model Brier:       {model_brier:.4f}")

Participant baseline Brier: 0.1920
Logistic model Brier:       0.1765


In [30]:
# Dynamic participant-specific nonresponse baseline
dynamic_baseline = (
    test_df["prior_cumulative_nonresponses"]
    / test_df["prior_cumulative_prompts"]
)

dynamic_baseline = dynamic_baseline.fillna(
    y_train_validation.mean()
)

dynamic_baseline_brier = brier_score_loss(
    y_test,
    dynamic_baseline
)

print(f"Dynamic historical baseline Brier: {dynamic_baseline_brier:.4f}")
print(f"Elastic Net logistic Brier:        {model_brier:.4f}")

Dynamic historical baseline Brier: 0.1880
Elastic Net logistic Brier:        0.1765


## Feature Selection

# Mixed Effect Logistic

In [31]:
TARGET = "nonresponse"

train_df = model_df[model_df["day"] <= 28].copy()

validation_df = model_df[
    (model_df["day"] >= 29) &
    (model_df["day"] <= 34)
].copy()

test_df = model_df[model_df["day"] >= 35].copy()

print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nNonresponse prevalence:")
print("Training:", train_df[TARGET].mean().round(4))
print("Validation:", validation_df[TARGET].mean().round(4))
print("Test:", test_df[TARGET].mean().round(4))

Training: (21000, 71)
Validation: (4500, 71)
Test: (4500, 71)

Nonresponse prevalence:
Training: 0.2753
Validation: 0.2993
Test: 0.2891


In [32]:
# Candidate predictors
FEATURE_SETS = {
    "core": [
        "day",
        "day_squared",
        "beep",
        "prior_7_day_response_rate",
        "days_since_previous_response",
    ],

    "same_beep": [
        "day",
        "day_squared",
        "beep",
        "prior_7_day_response_rate",
        "days_since_previous_response",
        "prior_same_beep_nonresponses",
    ],

    "combined": [
        "day",
        "day_squared",
        "beep",
        "prior_7_day_response_rate",
        "days_since_previous_response",
        "prior_same_beep_nonresponses",
        "prev_day_responded",
        "prev_response_same_day",
    ],
}

ALL_FEATURES = list(dict.fromkeys(
    feature
    for features in FEATURE_SETS.values()
    for feature in features
))

missing_features = [
    feature for feature in ALL_FEATURES
    if feature not in model_df.columns
]

if missing_features:
    raise ValueError(
        f"Missing predictors in model_df: {missing_features}"
    )

CONTINUOUS_FEATURES = [
    "day",
    "day_squared",
    "prior_7_day_response_rate",
    "days_since_previous_response",
    "prior_same_beep_nonresponses",
]

BINARY_FEATURES = [
    "prev_day_responded",
    "prev_response_same_day",
]

# Work with independent copies
train_model = train_df[["id", TARGET] + ALL_FEATURES].copy()
validation_model = validation_df[["id", TARGET] + ALL_FEATURES].copy()
test_model = test_df[["id", TARGET] + ALL_FEATURES].copy()

# Preserve participant identity
for data in [train_model, validation_model, test_model]:
    data["id"] = data["id"].astype(str)

# Beep is categorical, not a continuous numeric measurement
for data in [train_model, validation_model, test_model]:
    data["beep"] = pd.to_numeric(
        data["beep"], errors="raise"
    ).astype(int)

# Impute numeric features using training data only
IMPUTED_FEATURES = [
    feature for feature in ALL_FEATURES
    if feature != "beep"
]

imputer = SimpleImputer(strategy="median")

train_imputed = imputer.fit_transform(
    train_model[IMPUTED_FEATURES]
)

validation_imputed = imputer.transform(
    validation_model[IMPUTED_FEATURES]
)

test_imputed = imputer.transform(
    test_model[IMPUTED_FEATURES]
)

train_model[IMPUTED_FEATURES] = train_imputed
validation_model[IMPUTED_FEATURES] = validation_imputed
test_model[IMPUTED_FEATURES] = test_imputed

# Add missingness indicators only for predictors with
# missing observations in the training data.
MISSING_INDICATORS = []

for feature in IMPUTED_FEATURES:
    missing_column = f"{feature}_missing"

    if train_df[feature].isna().any():
        train_model[missing_column] = (
            train_df[feature].isna().astype(int).to_numpy()
        )

        validation_model[missing_column] = (
            validation_df[feature].isna().astype(int).to_numpy()
        )

        test_model[missing_column] = (
            test_df[feature].isna().astype(int).to_numpy()
        )

        if train_model[missing_column].nunique() > 1:
            MISSING_INDICATORS.append(missing_column)

# Standardize continuous predictors using training statistics
scaler = StandardScaler()

scaled_names = [
    f"{feature}_z" for feature in CONTINUOUS_FEATURES
]

train_model[scaled_names] = scaler.fit_transform(
    train_model[CONTINUOUS_FEATURES]
)

validation_model[scaled_names] = scaler.transform(
    validation_model[CONTINUOUS_FEATURES]
)

test_model[scaled_names] = scaler.transform(
    test_model[CONTINUOUS_FEATURES]
)

# Verify that no missing values remain in model predictors
print("Training missing values:")
print(train_model[ALL_FEATURES].isna().sum().sum())

print("\nTraining participants:", train_model["id"].nunique())
print("Validation participants:", validation_model["id"].nunique())
print("Test participants:", test_model["id"].nunique())

print("\nMissingness indicators:", MISSING_INDICATORS)

Training missing values:
0

Training participants: 125
Validation participants: 125
Test participants: 125

Missingness indicators: ['prior_7_day_response_rate_missing', 'days_since_previous_response_missing', 'prev_day_responded_missing', 'prev_response_same_day_missing']


In [33]:
def create_mixed_formula(features):
    formula_features = []

    for feature in features:

        if feature == "beep":
            formula_features.append("beep")

        elif feature in CONTINUOUS_FEATURES:
            formula_features.append(f"{feature}_z")

        else:
            formula_features.append(feature)

        missing_column = f"{feature}_missing"

        if missing_column in MISSING_INDICATORS:
            formula_features.append(missing_column)

    fixed_effects = " + ".join(formula_features)

    return f"{TARGET} ~ {fixed_effects} + (1 | id)"


FORMULAS = {
    name: create_mixed_formula(features)
    for name, features in FEATURE_SETS.items()
}

for name, formula in FORMULAS.items():
    print(f"\n{name.upper()}")
    print(formula)


CORE
nonresponse ~ day_z + day_squared_z + beep + prior_7_day_response_rate_z + prior_7_day_response_rate_missing + days_since_previous_response_z + days_since_previous_response_missing + (1 | id)

SAME_BEEP
nonresponse ~ day_z + day_squared_z + beep + prior_7_day_response_rate_z + prior_7_day_response_rate_missing + days_since_previous_response_z + days_since_previous_response_missing + prior_same_beep_nonresponses_z + (1 | id)

COMBINED
nonresponse ~ day_z + day_squared_z + beep + prior_7_day_response_rate_z + prior_7_day_response_rate_missing + days_since_previous_response_z + days_since_previous_response_missing + prior_same_beep_nonresponses_z + prev_day_responded + prev_day_responded_missing + prev_response_same_day + prev_response_same_day_missing + (1 | id)


In [34]:
def evaluate_probabilities(y_true, probabilities, name):
    probabilities = np.clip(
        np.asarray(probabilities),
        1e-7,
        1 - 1e-7
    )

    y_true = np.asarray(y_true)

    return {
        "model": name,
        "average_precision": average_precision_score(
            y_true, probabilities
        ),
        "roc_auc": roc_auc_score(
            y_true, probabilities
        ),
        "brier_score": brier_score_loss(
            y_true, probabilities
        ),
        "log_loss": log_loss(
            y_true, probabilities,
            labels=[0, 1]
        ),
    }


def predict_mixed_model(model, prediction_data):
    import inspect

    parameters = inspect.signature(
        model.predict
    ).parameters

    kwargs = {}

    if "use_rfx" in parameters:
        kwargs["use_rfx"] = True
    else:
        raise RuntimeError(
            "This Pymer4 predict method does not expose "
            "use_rfx. Check the installed API before predicting."
        )

    if "pred_type" in parameters:
        kwargs["pred_type"] = "response"

    probabilities = model.predict(
        prediction_data,
        **kwargs
    )

    return np.asarray(probabilities).reshape(-1)

In [37]:
print(train_model["beep"].dtype)
print(train_model["beep"].unique().tolist())

print("\nModel data:")
print(model.data["beep"].dtype)
print(model.data["beep"].unique().tolist())

int64
[1, 2, 3, 4, 5, 6]

Model data:
int64
[1, 2, 3, 4, 5, 6]


In [38]:
mixed_models = {}
validation_predictions = {}
validation_results = []

for name, formula in FORMULAS.items():

    print(f"\n{'=' * 65}")
    print(f"FITTING: {name.upper()}")
    print(f"{'=' * 65}")
    print(formula)

    # Make a separate copy so the original data remains unchanged.
    train_model_fit = train_model.copy()
    validation_model_fit = validation_model.copy()

    # Explicitly encode beep as a categorical predictor.
    train_model_fit["beep"] = pd.Categorical(
        train_model_fit["beep"],
        categories=[1, 2, 3, 4, 5, 6],
    )

    validation_model_fit["beep"] = pd.Categorical(
        validation_model_fit["beep"],
        categories=[1, 2, 3, 4, 5, 6],
    )

    model = Lmer(
        formula,
        data=train_model_fit,
        family="binomial",
    )

    model.fit(
        summarize=False,
        verbose=False,
    )

    probabilities = predict_mixed_model(
        model,
        validation_model_fit,
    )

    metrics = evaluate_probabilities(
        y_true=validation_model_fit[TARGET],
        probabilities=probabilities,
        name=name,
    )

    metrics["AIC"] = model.AIC
    metrics["log_likelihood"] = model.logLike

    validation_results.append(metrics)

    mixed_models[name] = model
    validation_predictions[name] = probabilities

    print("\nValidation metrics:")
    print(pd.DataFrame([metrics]).round(4))

    if getattr(model, "warnings", None):
        print("\nModel warnings:")
        print(model.warnings)


mixed_validation_results = pd.DataFrame(validation_results)

mixed_validation_results = mixed_validation_results.sort_values(
    ["average_precision", "log_loss"],
    ascending=[False, True],
)

display(mixed_validation_results.round(4))


FITTING: CORE
nonresponse ~ day_z + day_squared_z + beep + prior_7_day_response_rate_z + prior_7_day_response_rate_missing + days_since_previous_response_z + days_since_previous_response_missing + (1 | id)


AttributeError: 'NULLType' object has no attribute 'rx2'

In [39]:
print("Model class:", type(model))
print("AIC:", getattr(model, "AIC", None))
print("Log likelihood:", getattr(model, "logLike", None))
print("Coefficients:")
print(getattr(model, "coefs", None))

print("\nModel attributes:")
print([attribute for attribute in dir(model) if not attribute.startswith("_")])

Model class: <class 'pymer4.models.Lmer.Lmer'>
AIC: 22331.928501436458
Log likelihood: -11152.964250718229
Coefficients:
None

Model attributes:
['AIC', 'BIC', 'anova', 'coefs', 'confint', 'contrast_codes', 'contrasts', 'data', 'design_matrix', 'factors', 'factors_prev_', 'family', 'fit', 'fitted', 'fixef', 'formula', 'grps', 'logLike', 'marginal_contrasts', 'marginal_estimates', 'model_obj', 'plot', 'plot_summary', 'post_hoc', 'predict', 'ranef', 'ranef_corr', 'ranef_var', 'ranked_data', 'residuals', 'sig_type', 'simulate', 'summary', 'warnings']


In [40]:
from importlib.metadata import version
import inspect

print("Pymer4 version:", version("pymer4"))

source = inspect.getsource(Lmer.fit)
print("\n".join(source.splitlines()[475:500]))

Pymer4 version: 0.8.2
        if len(fixefs) > 1:
            if self.coefs is not None:
                f_corrected_order = []
                for f in fixefs:
                    f_corrected_order.append(
                        pd.DataFrame(
                            f[
                                list(self.coefs.index)
                                + [
                                    elem
                                    for elem in f.columns
                                    if elem not in self.coefs.index
                                ]
                            ]
                        )
                    )
                self.fixef = f_corrected_order
            else:
                self.fixef = list(fixefs)
        else:
            self.fixef = fixefs[0]
            if self.coefs is not None:
                self.fixef = self.fixef[
                    list(self.coefs.index)
                    + [


In [41]:
print("Fitted flag:", model.fitted)
print("R model object exists:", model.model_obj is not None)
print("Fitted values exist:", model.fits is not None)
print("Fixed effects:", model.fixef)
print("Random effects:", model.ranef)
print("Coefficients:", model.coefs)

Fitted flag: False
R model object exists: True


AttributeError: 'Lmer' object has no attribute 'fits'

In [42]:
print("Fitted flag:", model.fitted)
print("R model object exists:", model.model_obj is not None)

print("\nFixed effects:")
print(model.fixef)

print("\nRandom effects:")
print(model.ranef)

print("\nModel coefficients:")
print(model.coefs)

print("\nWarnings:")
print(model.warnings)

print("\nAvailable fitted-related attributes:")
print([
    attribute for attribute in dir(model)
    if any(term in attribute.lower() for term in ["fit", "predict", "resid", "model"])
])

Fitted flag: False
R model object exists: True

Fixed effects:
None

Random effects:
None

Model coefficients:
None

Warnings:
[]

Available fitted-related attributes:
['_refit_orthogonal', 'fit', 'fitted', 'model_obj', 'predict', 'residuals']


In [44]:
print("R model object:")
print(model.model_obj)

print("\nObject type:")
print(type(model.model_obj))

print("\nAvailable object attributes:")
print(dir(model.model_obj))

R model object:
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: 
nonresponse ~ day_z + day_squared_z + beep + prior_7_day_response_rate_z +  
    prior_7_day_response_rate_missing + days_since_previous_response_z +  
    days_since_previous_response_missing + (1 | id)
   Data: structure(list(id = c("1", "1", "1", "1", "1", "1", "1", "1",  
"1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1",  
"1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1",  
"1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1",  
"1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1",  
"1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1",  
"1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1",  
"1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1",  
"1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1", "1",  
"1", "1", "1", "1", "1", "1", "1", "1"

In [45]:
import pymer4.utils as utils

print([
    name for name in dir(utils)
    if not name.startswith("_")
])

['LogisticRegression', 'MAX_INT', 'base', 'chi2', 'dmatrices', 'get_resource_path', 'importr', 'isPSD', 'nearestPSD', 'norm', 'np', 'os', 'pd', 'result_to_table', 'upper']


In [47]:
import rpy2.robjects as ro

# Retrieve R functions directly from their package namespaces
fixef = ro.r("lme4::fixef")
ranef = ro.r("lme4::ranef")
fitted = ro.r("stats::fitted")
is_singular = ro.r("lme4::isSingular")

# Extract estimates
fixed_effects = fixef(model.model_obj)
random_effects = ranef(model.model_obj)
fitted_probabilities = fitted(model.model_obj)

print("FIXED EFFECTS")
print(fixed_effects)

print("\nRANDOM EFFECTS")
print(random_effects)

print("\nFIRST 10 FITTED PROBABILITIES")
print(list(fitted_probabilities)[:10])

print("\nSINGULAR FIT")
print(is_singular(model.model_obj))

FIXED EFFECTS
                         (Intercept)                                day_z 
                          -1.8429401                            0.2407088 
                       day_squared_z                                beep2 
                          -0.1740188                            0.9791002 
                               beep3                                beep4 
                           0.6507443                            0.8250790 
                               beep5                                beep6 
                           0.8721488                            1.1011479 
         prior_7_day_response_rate_z    prior_7_day_response_rate_missing 
                          -0.1838855                            0.1534046 
      days_since_previous_response_z days_since_previous_response_missing 
                           0.4940422                            1.0864488 


RANDOM EFFECTS
$id
    (Intercept)
1    0.08243845
10   0.17367418
100  0.41311600
1

In [48]:
import rpy2.robjects as ro

print("Optimizer diagnostics:")
print(model.model_obj.do_slot("optinfo"))

print("\nFull R model summary:")
print(ro.r("summary")(model.model_obj))

print("\nRandom-effect variance:")
print(ro.r("lme4::VarCorr")(model.model_obj))

Optimizer diagnostics:
$optimizer
[1] "Nelder_Mead"

$control
$control$xst
 [1] 0.020000000 0.014794359 0.015628277 0.015097539 0.013281243 0.014319229
 [7] 0.014753308 0.014919731 0.014890974 0.005109138 0.023578952 0.005678512
[13] 0.038722554

$control$xt
 [1] 1.000000e-05 7.397179e-06 7.814138e-06 7.548769e-06 6.640622e-06
 [6] 7.159615e-06 7.376654e-06 7.459865e-06 7.445487e-06 2.554569e-06
[11] 1.178948e-05 2.839256e-06 1.936128e-05

$control$verbose
[1] 0


$derivs
NULL

$conv
$conv$opt
[1] 0

$conv$lme4
NULL


$feval
[1] 2229

$message
[1] "parameter values converged to within tolerance"

$warnings
list()

$val
 [1]  0.5544326 -1.8429401  0.2407088 -0.1740188  0.9791002  0.6507443
 [7]  0.8250790  0.8721488  1.1011479 -0.1838855  0.1534046  0.4940422
[13]  1.0864488



Full R model summary:
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: 
nonresponse ~ day_z + day_squared_z + beep + prior_7_day_

In [49]:
import rpy2.robjects as ro

diagnostics = ro.r("""
function(x) {
    opt <- x@optinfo

    list(
        optimizer = opt$optimizer,
        convergence_code = opt$conv$opt,
        convergence_messages = opt$conv$lme4$messages,
        optimizer_warnings = opt$warnings,
        max_absolute_gradient = if (!is.null(opt$derivs$gradient)) {
            max(abs(opt$derivs$gradient))
        } else {
            NA_real_
        }
    )
}
""")

print(diagnostics(model.model_obj))

$optimizer
[1] "Nelder_Mead"

$convergence_code
[1] 0

$convergence_messages
NULL

$optimizer_warnings
list()

$max_absolute_gradient
[1] NA




In [50]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter

predict_validation = ro.r("""
function(model, newdata) {
    predict(
        model,
        newdata = newdata,
        type = "response",
        re.form = NULL,
        allow.new.levels = FALSE
    )
}
""")

with localconverter(default_converter + pandas2ri.converter):
    r_validation = ro.conversion.py2rpy(validation_model_fit)

validation_probabilities = np.asarray(
    predict_validation(model.model_obj, r_validation)
).flatten()

print("Predictions:", len(validation_probabilities))
print("Mean predicted probability:", validation_probabilities.mean())
print("Minimum:", validation_probabilities.min())
print("Maximum:", validation_probabilities.max())

Predictions: 4500
Mean predicted probability: 0.2798698203284815
Minimum: 0.05687006019301863
Maximum: 0.949488110863447


In [52]:
mixed_metrics = evaluate_probabilities(
    y_true=validation_model_fit[TARGET],
    probabilities=validation_probabilities,
    name="GLMM Random Intercept",
)

display(pd.DataFrame([mixed_metrics]).round(4))

,model,average_precision,roc_auc,brier_score,log_loss
0,GLMM Random Intercept,0.5389,0.7249,0.1811,0.5416


In [53]:
# Generate population-level predictions (fixed effects only)

predict_population = ro.r("""
function(model, newdata) {
    predict(
        model,
        newdata = newdata,
        type = "response",
        re.form = NA,
        allow.new.levels = TRUE
    )
}
""")

population_probabilities = np.asarray(
    predict_population(model.model_obj, r_validation)
).flatten()

print("Predictions:", len(population_probabilities))
print("Mean predicted probability:", population_probabilities.mean())
print("Minimum:", population_probabilities.min())
print("Maximum:", population_probabilities.max())

Predictions: 4500
Mean predicted probability: 0.26058879793771755
Minimum: 0.1484709072651926
Maximum: 0.9393232240702389


In [54]:
population_metrics = evaluate_probabilities(
    y_true=validation_model_fit[TARGET],
    probabilities=population_probabilities,
    name="GLMM Population-Level",
)

display(pd.DataFrame([population_metrics]).round(4))

,model,average_precision,roc_auc,brier_score,log_loss
0,GLMM Population-Level,0.5167,0.6973,0.1926,0.57


In [55]:
comparison = pd.DataFrame([
    {
        "model": "Elastic Net",
        "average_precision": 0.568351,
        "brier_score": 0.175793,
        "log_loss": 0.530034,
    },
    {
        "model": "GLMM Conditional",
        "average_precision": 0.5389,
        "brier_score": 0.1811,
        "log_loss": 0.5416,
    },
    {
        "model": "GLMM Population-Level",
        "average_precision": population_metrics["average_precision"],
        "brier_score": population_metrics["brier_score"],
        "log_loss": population_metrics["log_loss"],
    },
])

display(
    comparison.sort_values(
        "average_precision",
        ascending=False,
    ).round(4)
)

,model,average_precision,brier_score,log_loss
0,Elastic Net,0.5684,0.1758,0.5300
1,GLMM Conditional,0.5389,0.1811,0.5416
2,GLMM Population-Level,0.5167,0.1926,0.5700


In [56]:
random_slope_formula = """
nonresponse ~ day_z +
    day_squared_z +
    beep +
    prior_7_day_response_rate_z +
    prior_7_day_response_rate_missing +
    days_since_previous_response_z +
    days_since_previous_response_missing +
    (1 + day_z | id)
"""

In [57]:
random_slope_model = Lmer(
    random_slope_formula,
    data=train_model_fit,
    family="binomial",
)

try:
    random_slope_model.fit(
        summarize=False,
        verbose=False,
    )
except AttributeError as exc:
    # Pymer4 0.8.2 can raise this after R has successfully fitted
    # the model because of its post-fit NULL diagnostic handling.
    if random_slope_model.model_obj is None:
        raise

    print("R model fitted; Pymer4 post-fit processing raised:")
    print(exc)

R model fitted; Pymer4 post-fit processing raised:
'NULLType' object has no attribute 'rx2'


In [58]:
is_singular = ro.r("lme4::isSingular")
get_variance = ro.r("lme4::VarCorr")

print("Singular:", is_singular(random_slope_model.model_obj))
print("\nRandom effects:")
print(get_variance(random_slope_model.model_obj))

print("\nConvergence diagnostics:")
print(diagnostics(random_slope_model.model_obj))

Singular: [1] FALSE


Random effects:
 Groups Name        Std.Dev. Corr  
 id     (Intercept) 0.66613        
        day_z       0.20923  0.177 


Convergence diagnostics:
$optimizer
[1] "Nelder_Mead"

$convergence_code
[1] 0

$convergence_messages
NULL

$optimizer_warnings
list()

$max_absolute_gradient
[1] NA




In [59]:
random_slope_conditional = np.asarray(
    predict_validation(
        random_slope_model.model_obj,
        r_validation,
    )
).flatten()

random_slope_population = np.asarray(
    predict_population(
        random_slope_model.model_obj,
        r_validation,
    )
).flatten()

In [60]:
random_slope_conditional_metrics = evaluate_probabilities(
    y_true=validation_model_fit[TARGET],
    probabilities=random_slope_conditional,
    name="GLMM Random Intercept + Day Slope",
)

random_slope_population_metrics = evaluate_probabilities(
    y_true=validation_model_fit[TARGET],
    probabilities=random_slope_population,
    name="GLMM Day Slope Population-Level",
)

display(pd.DataFrame([
    random_slope_conditional_metrics,
    random_slope_population_metrics,
]).round(4))

,model,average_precision,roc_auc,brier_score,log_loss
0,GLMM Random Intercept + Day Slope,0.5412,0.7228,0.1808,0.5427
1,GLMM Day Slope Population-Level,0.4746,0.6465,0.1994,0.5869
